# Stroke Guide Extraction Sandbox

This notebook mirrors the production stroke-extraction pipeline in `image_processing_service.py`.
It follows the same method as `stroke_extraction.ipynb`: binarize, skeletonize, walk ordered strokes,
sample guide dots by arc length, and render one transparent PNG per stroke.

In [ ]:
from math import ceil
from pathlib import Path
import base64
import io

import cv2
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

from image_processing_service import extract_stroke_guides

IMAGE_PATH = Path("image.png")
OUTPUT_DIR = Path("../generated_strokes_notebook")
MAX_PREVIEW_STROKES = 12


In [ ]:
image = Image.open(IMAGE_PATH).convert("RGB")
result = extract_stroke_guides(image, output_dir=OUTPUT_DIR)

overlay = np.asarray(image).copy()
for stroke in result.strokes:
    points = stroke.points
    for left, right in zip(points, points[1:]):
        cv2.line(overlay, left, right, (167, 139, 250), 1, cv2.LINE_AA)
    for index, point in enumerate(points):
        radius = 6 if index == 0 else 4
        color = (99, 102, 241) if index == 0 else (124, 58, 237)
        cv2.circle(overlay, point, radius, color, -1, cv2.LINE_AA)
        cv2.circle(overlay, point, radius, (76, 29, 149), 1, cv2.LINE_AA)

fig, axes = plt.subplots(1, 4, figsize=(24, 6))
axes[0].imshow(image)
axes[0].set_title("Original")
axes[1].imshow(result.binary_mask, cmap="gray")
axes[1].set_title(f"Binary ({result.mode})")
axes[2].imshow(result.skeleton, cmap="gray")
axes[2].set_title("Skeleton")
axes[3].imshow(overlay)
axes[3].set_title(f"Overlay: {len(result.strokes)} strokes / {result.total_dots} dots")
for axis in axes:
    axis.axis("off")
plt.tight_layout()
plt.show()

print(f"Mode: {result.mode}")
print(f"Saved stroke PNGs to: {OUTPUT_DIR.resolve()}")
print(f"Stroke count: {len(result.strokes)}")
print(f"Total dots: {result.total_dots}")
for stroke in result.strokes[:5]:
    print(
        f"stroke={stroke.stroke_id:>3} len={stroke.stroke_len_px:>4}px dots={len(stroke.points):>3} "
        f"start={stroke.points[0]} end={stroke.points[-1]}"
    )


In [ ]:
def decode_data_url(data_url: str) -> np.ndarray:
    _, payload = data_url.split(",", 1)
    return np.asarray(Image.open(io.BytesIO(base64.b64decode(payload))).convert("RGBA"))

stroke_layers = [decode_data_url(stroke.data_url) for stroke in result.strokes[:MAX_PREVIEW_STROKES]]

cols = 4
rows = max(1, ceil(len(stroke_layers) / cols))
fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
axes = np.atleast_1d(axes).reshape(rows, cols)

for index, layer in enumerate(stroke_layers):
    row, col = divmod(index, cols)
    axes[row, col].imshow(layer)
    axes[row, col].set_title(f"Stroke {index} ({int((layer[..., 3] > 0).sum()):,} px)")
    axes[row, col].axis("off")

for index in range(len(stroke_layers), rows * cols):
    row, col = divmod(index, cols)
    axes[row, col].axis("off")

plt.tight_layout()
plt.show()
